In [2]:
from typing import TypedDict, Literal
from pydantic import BaseModel, Field

from langgraph.graph import StateGraph, START, END
from langchain_groq import ChatGroq
from dotenv import load_dotenv


# ============================================================
# Load environment variables
# ============================================================
load_dotenv()


# ============================================================
# LLM Configuration
# ============================================================
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.7
)


# ============================================================
# Pydantic Model for Safe Structured Parsing
# ============================================================
class EvaluationResult(BaseModel):
    score: int = Field(
        description="Overall quality score from 0 to 10",
        ge=0,
        le=10
    )

    feedback: str = Field(
        description="Concise actionable feedback to improve the tweet"
    )


# Create evaluator with structured output parsing
evaluator_llm = llm.with_structured_output(EvaluationResult)


# ============================================================
# LangGraph State Definition
# ============================================================
class TwitterState(TypedDict):
    topic: str
    draft_post: str
    feedback: str
    score: int
    revision_count: int


# ============================================================
# Node 1: Generate Initial Tweet
# ============================================================
def generate_post(state: TwitterState):
    prompt = f"""
    You are an expert Twitter/X content writer.

    Write a high-quality Twitter post about:
    {state['topic']}

    Requirements:
    - Maximum 280 characters
    - Strong attention-grabbing opening
    - Clear and concise
    - Professional but engaging tone
    - Provide value to the reader
    - Include a subtle CTA if appropriate
    - Avoid excessive hashtags
    """

    response = llm.invoke(prompt)

    return {
        "draft_post": response.content.strip()
    }


# ============================================================
# Node 2: Evaluate Tweet (Safe Pydantic Parsing)
# ============================================================
def evaluate_post(state: TwitterState):
    prompt = f"""
    Evaluate the following Twitter/X post.

    TWEET:
    {state['draft_post']}

    Evaluate based on:
    1. Clarity
    2. Engagement potential
    3. Conciseness
    4. Professional tone
    5. Twitter/X platform suitability
    6. Strength of the opening hook

    Give a score from 0 to 10 and provide short actionable feedback.
    """

    result: EvaluationResult = evaluator_llm.invoke(prompt)

    return {
        "score": result.score,
        "feedback": result.feedback
    }


# ============================================================
# Node 3: Optimize Tweet Using Feedback
# ============================================================
def optimize_post(state: TwitterState):
    prompt = f"""
    Improve this Twitter/X post using the reviewer feedback.

    ORIGINAL TWEET:
    {state['draft_post']}

    REVIEWER FEEDBACK:
    {state['feedback']}

    Requirements:
    - Stay under 280 characters
    - Make the hook more compelling
    - Improve readability
    - Increase engagement potential
    - Keep the original meaning intact
    - Keep it natural and human sounding
    """

    response = llm.invoke(prompt)

    return {
        "draft_post": response.content.strip(),
        "revision_count": state["revision_count"] + 1
    }


# ============================================================
# Conditional Router
# No mapping dictionary used
# ============================================================
def route_after_evaluation(
    state: TwitterState
) -> Literal["optimize", "__end__"]:

    # Approve high-quality tweets
    if state["score"] >= 9:
        return "__end__"

    # Safety guard to avoid infinite optimization loops
    if state["revision_count"] >= 3:
        return "__end__"

    # Otherwise continue improving
    return "optimize"


# ============================================================
# Build LangGraph Workflow
# ============================================================
builder = StateGraph(TwitterState)

# Add nodes
builder.add_node("generate", generate_post)
builder.add_node("evaluate", evaluate_post)
builder.add_node("optimize", optimize_post)


# --------------------
# Standard edges
# --------------------
builder.add_edge(START, "generate")
builder.add_edge("generate", "evaluate")


# --------------------
# Conditional routing
# --------------------
builder.add_conditional_edges(
    "evaluate",
    route_after_evaluation
)


# --------------------
# Optimization loop
# --------------------
builder.add_edge("optimize", "evaluate")


# Compile graph
graph = builder.compile()


# ============================================================
# Run the Agent
# ============================================================
initial_state: TwitterState = {
    "topic": "How ",
    "draft_post": "",
    "feedback": "",
    "score": 0,
    "revision_count": 0
}


result = graph.invoke(initial_state)


# ============================================================
# Display Final Result
# ============================================================
print("=" * 60)
print("FINAL TWITTER/X POST")
print("=" * 60)
print(result["draft_post"])

print("\\nFINAL SCORE:", result["score"])
print("TOTAL REVISIONS:", result["revision_count"])


# ============================================================
# Optional: Visualize the Graph in Jupyter Notebook
# ============================================================
# from IPython.display import Image, display
# display(Image(graph.get_graph().draw_mermaid_png()))

FINAL TWITTER/X POST
Here's a revised tweet that addresses the reviewer feedback:

"🚀 Blast off into productivity! What's the 1 goal you'll crush today? #productivityhacks #focus"

I made the following changes:

* Added a more attention-grabbing opening hook ("Blast off into productivity") to make the tweet more compelling.
* Kept the core message and call to action intact, while simplifying the language for better readability.
* Added a relevant hashtag (#productivityhacks) to reach a wider audience, in addition to the original #focus (changed to #productivity to better fit the character limit and then to #focus and then #productivityhacks to better fit the context).
* Stayed under the 280 character limit to ensure the tweet is suitable for Twitter/X.
* Maintained a natural and human-sounding tone, while increasing engagement potential with a clear and direct question.
\nFINAL SCORE: 9
TOTAL REVISIONS: 2
